STAGE 5.2
# HAND DETECTOR / ROI GENERATOR SELECTION

In [13]:
# =============================================================================
# CISLR — STAGE 5.2
# FINAL CORRECTED HAND DETECTOR / ROI GENERATOR SELECTION
# =============================================================================
#
# FINAL FIX:
# MediaPipe VIDEO mode requires monotonically increasing timestamps for a
# continuous sequence. A fresh HandLandmarker instance is therefore created
# for EACH independent video.
#
# Everything else remains controlled:
#   - Same deterministic 200-video subset
#   - Same 8 uniform frames/video
#   - Same candidate detectors
#   - Same ROI validity criteria
#   - Same evaluation metrics
#   - Same selection weights
#   - Same OpenCV implementation
#
# DO NOT RERUN:
#   Stage 4.1 → Stage 4.9
#   Stage 5.1
#   MobileNet training
#   Official retrieval evaluation
#
# Raw CISLR videos are never modified.
# =============================================================================


import os
import sys
import cv2
import json
import time
import math
import random
import hashlib
import subprocess
import importlib.util
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# =============================================================================
# 0. CONFIGURATION
# =============================================================================

PROJECT_ROOT = Path(
    "/home/dipshikha/Music/cao/CISLR_RESEARCH"
)

STAGE5_ROOT = (
    PROJECT_ROOT
    / "audit"
    / "stage5_hand_roi"
)

MANIFEST_PATH = (
    STAGE5_ROOT
    / "manifests"
    / "stage5_1_master_video_manifest.csv"
)

OUTPUT_ROOT = (
    STAGE5_ROOT
    / "detector_selection"
)

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

# -------------------------------------------------------------------------
# Output files
# -------------------------------------------------------------------------

RESULTS_CSV = (
    OUTPUT_ROOT
    / "stage5_2_detector_comparison.csv"
)

FRAME_RESULTS_CSV = (
    OUTPUT_ROOT
    / "stage5_2_frame_level_results.csv"
)

PROTOCOL_JSON = (
    OUTPUT_ROOT
    / "stage5_2_detector_selection_protocol.json"
)

SUMMARY_JSON = (
    OUTPUT_ROOT
    / "stage5_2_detector_selection_summary.json"
)

SELECTION_MANIFEST_PATH = (
    OUTPUT_ROOT
    / "stage5_2_selection_video_manifest.csv"
)

ERROR_LOG_CSV = (
    OUTPUT_ROOT
    / "stage5_2_detector_errors.csv"
)

VISUAL_DIR = (
    OUTPUT_ROOT
    / "visual_sanity_check"
)

VISUAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# =============================================================================
# REPRODUCIBILITY
# =============================================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)


# =============================================================================
# DATASET SAMPLING
# =============================================================================

N_SELECTION_VIDEOS = 200

FRAMES_PER_VIDEO = 8

N_VISUAL_VIDEOS = 12


# =============================================================================
# ROI VALIDITY
# =============================================================================

MIN_ROI_AREA_FRACTION = 0.005

MAX_ROI_AREA_FRACTION = 0.90


# =============================================================================
# MEDIAPIPE
# =============================================================================

MEDIAPIPE_MAX_HANDS = 2

MEDIAPIPE_MIN_DETECTION_CONFIDENCE = 0.50

MEDIAPIPE_MIN_PRESENCE_CONFIDENCE = 0.50

MEDIAPIPE_MIN_TRACKING_CONFIDENCE = 0.50


# =============================================================================
# OPENCV SKIN / CONTOUR
# =============================================================================

HSV_LOWER = np.array(
    [0, 20, 40],
    dtype=np.uint8
)

HSV_UPPER = np.array(
    [35, 255, 255],
    dtype=np.uint8
)

YCRCB_LOWER = np.array(
    [0, 133, 77],
    dtype=np.uint8
)

YCRCB_UPPER = np.array(
    [255, 173, 127],
    dtype=np.uint8
)

MORPH_KERNEL_SIZE = 5

MIN_CONTOUR_AREA = 150


# =============================================================================
# MEDIAPIPE MODEL
# =============================================================================

MEDIAPIPE_MODEL_DIR = (
    OUTPUT_ROOT
    / "models"
)

MEDIAPIPE_MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

HAND_MODEL_PATH = (
    MEDIAPIPE_MODEL_DIR
    / "hand_landmarker.task"
)

HAND_MODEL_URL = (
    "https://storage.googleapis.com/"
    "mediapipe-models/"
    "hand_landmarker/"
    "hand_landmarker/"
    "float16/1/"
    "hand_landmarker.task"
)


# =============================================================================
# 1. START
# =============================================================================

print("=" * 80)
print("CISLR — STAGE 5.2")
print("FINAL CORRECTED HAND DETECTOR / ROI GENERATOR SELECTION")
print("=" * 80)

print(
    "\nProject root:",
    PROJECT_ROOT
)

if not PROJECT_ROOT.exists():

    raise FileNotFoundError(
        f"Project root not found:\n{PROJECT_ROOT}"
    )

if not MANIFEST_PATH.exists():

    raise FileNotFoundError(
        f"Stage 5.1 manifest not found:\n{MANIFEST_PATH}"
    )


# =============================================================================
# 2. LOAD STAGE 5.1 MASTER MANIFEST
# =============================================================================

manifest = pd.read_csv(
    MANIFEST_PATH
)

print(
    "\nManifest shape:",
    manifest.shape
)

required_columns = [
    "uid",
    "gloss",
    "video_path"
]

missing_columns = [
    c
    for c in required_columns
    if c not in manifest.columns
]

if missing_columns:

    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )


manifest = (
    manifest
    .drop_duplicates(
        subset=["uid"]
    )
    .reset_index(
        drop=True
    )
)

print(
    "Unique videos:",
    len(manifest)
)


# =============================================================================
# 3. VERIFY VIDEO PATHS
# =============================================================================

manifest["video_exists"] = (
    manifest["video_path"]
    .apply(
        lambda p: Path(
            str(p)
        ).exists()
    )
)

missing_videos = int(
    (~manifest["video_exists"])
    .sum()
)

print(
    "Videos found:",
    int(
        manifest["video_exists"]
        .sum()
    )
)

print(
    "Videos missing:",
    missing_videos
)

if missing_videos > 0:

    print(
        "\nWARNING — missing video examples:"
    )

    print(
        manifest.loc[
            ~manifest["video_exists"],
            [
                "uid",
                "video_path"
            ]
        ].head(10)
    )


# =============================================================================
# 4. RECREATE THE SAME DETERMINISTIC 200-VIDEO SUBSET
# =============================================================================
#
# This reproduces the original selection using the exact SHA256 UID ordering.
# It should therefore be identical to the previous Stage 5.2 subset.
# =============================================================================

valid_manifest = (
    manifest[
        manifest["video_exists"]
    ]
    .copy()
)


if len(valid_manifest) < N_SELECTION_VIDEOS:

    raise RuntimeError(
        f"Only {len(valid_manifest)} valid videos available."
    )


def deterministic_uid_hash(uid):

    return int(
        hashlib.sha256(
            str(uid)
            .encode("utf-8")
        )
        .hexdigest()[:16],
        16
    )


valid_manifest["_uid_hash"] = (
    valid_manifest["uid"]
    .apply(
        deterministic_uid_hash
    )
)

selection_manifest = (
    valid_manifest
    .sort_values(
        "_uid_hash"
    )
    .head(
        N_SELECTION_VIDEOS
    )
    .drop(
        columns=[
            "_uid_hash",
            "video_exists"
        ]
    )
    .reset_index(
        drop=True
    )
)

selection_manifest.to_csv(
    SELECTION_MANIFEST_PATH,
    index=False
)

print(
    "\nFixed detector-selection subset:",
    len(selection_manifest),
    "videos"
)

print(
    "Saved:",
    SELECTION_MANIFEST_PATH
)


# =============================================================================
# 5. VERIFY SUBSET AGAINST EXISTING STAGE 5.2 MANIFEST
# =============================================================================
#
# This protects the controlled comparison.
# =============================================================================

previous_manifest_exists = (
    SELECTION_MANIFEST_PATH.exists()
)

if previous_manifest_exists:

    previous_manifest = pd.read_csv(
        SELECTION_MANIFEST_PATH
    )

    previous_uids = list(
        previous_manifest[
            "uid"
        ].astype(str)
    )

    current_uids = list(
        selection_manifest[
            "uid"
        ].astype(str)
    )

    if previous_uids == current_uids:

        print(
            "Subset verification: IDENTICAL"
        )

    else:

        print(
            "WARNING: subset ordering differs."
        )


# =============================================================================
# 6. IMPORT MODERN MEDIAPIPE
# =============================================================================

print(
    "\nChecking MediaPipe..."
)

if importlib.util.find_spec(
    "mediapipe"
) is None:

    print(
        "MediaPipe not found. Installing..."
    )

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "mediapipe"
    ])

import mediapipe as mp

from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision

print(
    "MediaPipe version:",
    getattr(
        mp,
        "__version__",
        "unknown"
    )
)


# =============================================================================
# 7. DOWNLOAD / VERIFY HAND LANDMARKER MODEL
# =============================================================================

if not HAND_MODEL_PATH.exists():

    print(
        "\nDownloading MediaPipe Hand Landmarker model..."
    )

    urllib.request.urlretrieve(
        HAND_MODEL_URL,
        HAND_MODEL_PATH
    )

    print(
        "Download complete."
    )

else:

    print(
        "\nHand Landmarker model already exists."
    )

print(
    "Model:",
    HAND_MODEL_PATH
)


# =============================================================================
# 8. MEDIAPIPE LANDMARKER FACTORY
# =============================================================================
#
# CRITICAL FIX:
#
# A NEW HandLandmarker is created for each video.
#
# Therefore each video can safely use:
#
#       timestamp = 0, 100, 200, ...
#
# without violating MediaPipe's monotonically increasing timestamp
# requirement.
# =============================================================================

def create_mediapipe_landmarker():

    base_options = (
        mp_python.BaseOptions(
            model_asset_path=str(
                HAND_MODEL_PATH
            )
        )
    )

    options = (
        vision.HandLandmarkerOptions(

            base_options=base_options,

            running_mode=(
                vision.RunningMode.VIDEO
            ),

            num_hands=(
                MEDIAPIPE_MAX_HANDS
            ),

            min_hand_detection_confidence=(
                MEDIAPIPE_MIN_DETECTION_CONFIDENCE
            ),

            min_hand_presence_confidence=(
                MEDIAPIPE_MIN_PRESENCE_CONFIDENCE
            ),

            min_tracking_confidence=(
                MEDIAPIPE_MIN_TRACKING_CONFIDENCE
            )
        )
    )

    return (
        vision.HandLandmarker
        .create_from_options(
            options
        )
    )


# Test initialization once.
test_landmarker = (
    create_mediapipe_landmarker()
)

test_landmarker.close()

print(
    "MediaPipe Hand Landmarker initialization: PASS"
)


# =============================================================================
# 9. HELPER FUNCTIONS
# =============================================================================

def uniform_frame_indices(
    frame_count,
    n_frames=8
):

    if frame_count <= 0:

        return []

    if frame_count <= n_frames:

        return (
            np.linspace(
                0,
                max(
                    frame_count - 1,
                    0
                ),
                frame_count,
                dtype=int
            )
            .tolist()
        )

    return (
        np.linspace(
            0,
            frame_count - 1,
            n_frames,
            dtype=int
        )
        .tolist()
    )


def clamp_box(
    x1,
    y1,
    x2,
    y2,
    width,
    height
):

    x1 = max(
        0,
        min(
            int(round(x1)),
            width - 1
        )
    )

    y1 = max(
        0,
        min(
            int(round(y1)),
            height - 1
        )
    )

    x2 = max(
        0,
        min(
            int(round(x2)),
            width - 1
        )
    )

    y2 = max(
        0,
        min(
            int(round(y2)),
            height - 1
        )
    )

    if (
        x2 <= x1
        or
        y2 <= y1
    ):

        return None

    return (
        x1,
        y1,
        x2,
        y2
    )


def box_area_fraction(
    box,
    width,
    height
):

    if box is None:

        return 0.0

    x1, y1, x2, y2 = box

    area = (
        max(
            0,
            x2 - x1
        )
        *
        max(
            0,
            y2 - y1
        )
    )

    total = (
        width
        *
        height
    )

    if total <= 0:

        return 0.0

    return float(
        area / total
    )


def box_center(box):

    if box is None:

        return None

    x1, y1, x2, y2 = box

    return (
        (x1 + x2) / 2.0,
        (y1 + y2) / 2.0
    )


def center_distance(
    box_a,
    box_b,
    width,
    height
):

    ca = box_center(
        box_a
    )

    cb = box_center(
        box_b
    )

    if (
        ca is None
        or
        cb is None
    ):

        return np.nan

    dx = (
        ca[0] - cb[0]
    ) / max(
        width,
        1
    )

    dy = (
        ca[1] - cb[1]
    ) / max(
        height,
        1
    )

    return float(
        math.sqrt(
            dx * dx
            +
            dy * dy
        )
    )


def merge_boxes(
    boxes,
    width,
    height
):

    valid = [
        b
        for b in boxes
        if b is not None
    ]

    if not valid:

        return None

    x1 = min(
        b[0]
        for b in valid
    )

    y1 = min(
        b[1]
        for b in valid
    )

    x2 = max(
        b[2]
        for b in valid
    )

    y2 = max(
        b[3]
        for b in valid
    )

    return clamp_box(
        x1,
        y1,
        x2,
        y2,
        width,
        height
    )


# =============================================================================
# 10. MEDIAPIPE DETECTOR
# =============================================================================
#
# NOTE:
# This function receives a detector instance.
# The detector instance belongs to ONE video only.
# =============================================================================

def detect_mediapipe(
    landmarker,
    frame,
    timestamp_ms
):

    height, width = (
        frame.shape[:2]
    )

    rgb = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2RGB
    )

    # Correct modern MediaPipe image construction.
    mp_image = mp.Image(
        image_format=(
            mp.ImageFormat.SRGB
        ),
        data=rgb
    )

    result = (
        landmarker
        .detect_for_video(
            mp_image,
            int(timestamp_ms)
        )
    )

    boxes = []

    confidences = []

    if result.hand_landmarks:

        for hand_idx, landmarks in enumerate(
            result.hand_landmarks
        ):

            xs = [
                lm.x * width
                for lm in landmarks
            ]

            ys = [
                lm.y * height
                for lm in landmarks
            ]

            x1 = min(xs)
            y1 = min(ys)

            x2 = max(xs)
            y2 = max(ys)

            # Add modest context around landmark extent.
            bw = x2 - x1
            bh = y2 - y1

            margin_x = (
                0.20 * bw
            )

            margin_y = (
                0.20 * bh
            )

            box = clamp_box(
                x1 - margin_x,
                y1 - margin_y,
                x2 + margin_x,
                y2 + margin_y,
                width,
                height
            )

            if box is not None:

                boxes.append(
                    box
                )

            # Handedness score is recorded as a confidence proxy.
            try:

                score = (
                    result
                    .handedness[
                        hand_idx
                    ][0]
                    .score
                )

                confidences.append(
                    float(score)
                )

            except Exception:

                pass

    roi_box = merge_boxes(
        boxes,
        width,
        height
    )

    return {

        "boxes": boxes,

        "confidence": (
            float(
                np.mean(
                    confidences
                )
            )
            if confidences
            else 0.0
        ),

        "num_hands": len(
            boxes
        ),

        "roi_box": roi_box
    }


# =============================================================================
# 11. OPENCV SKIN / CONTOUR DETECTOR
# =============================================================================

def detect_opencv_skin(
    frame
):

    height, width = (
        frame.shape[:2]
    )

    hsv = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2HSV
    )

    ycrcb = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2YCrCb
    )

    mask_hsv = cv2.inRange(
        hsv,
        HSV_LOWER,
        HSV_UPPER
    )

    mask_ycrcb = cv2.inRange(
        ycrcb,
        YCRCB_LOWER,
        YCRCB_UPPER
    )

    mask = cv2.bitwise_and(
        mask_hsv,
        mask_ycrcb
    )

    kernel = np.ones(
        (
            MORPH_KERNEL_SIZE,
            MORPH_KERNEL_SIZE
        ),
        dtype=np.uint8
    )

    mask = cv2.morphologyEx(
        mask,
        cv2.MORPH_OPEN,
        kernel
    )

    mask = cv2.morphologyEx(
        mask,
        cv2.MORPH_CLOSE,
        kernel
    )

    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    candidate_boxes = []

    for contour in contours:

        area = cv2.contourArea(
            contour
        )

        if (
            area
            <
            MIN_CONTOUR_AREA
        ):

            continue

        x, y, w, h = (
            cv2.boundingRect(
                contour
            )
        )

        if (
            w < 10
            or
            h < 10
        ):

            continue

        box = clamp_box(
            x,
            y,
            x + w,
            y + h,
            width,
            height
        )

        if box is not None:

            candidate_boxes.append(
                (
                    box,
                    area
                )
            )

    if not candidate_boxes:

        return {

            "boxes": [],

            "confidence": 0.0,

            "num_hands": 0,

            "roi_box": None
        }

    candidate_boxes.sort(
        key=lambda x: x[1],
        reverse=True
    )

    selected = (
        candidate_boxes[:2]
    )

    boxes = [
        item[0]
        for item in selected
    ]

    roi_box = merge_boxes(
        boxes,
        width,
        height
    )

    largest_area = (
        selected[0][1]
    )

    confidence_proxy = min(
        largest_area
        /
        float(
            width * height
        ),
        1.0
    )

    return {

        "boxes": boxes,

        "confidence": float(
            confidence_proxy
        ),

        "num_hands": len(
            boxes
        ),

        "roi_box": roi_box
    }


# =============================================================================
# 12. ROBUST VIDEO DECODER
# =============================================================================

def read_uniform_frames(
    video_path,
    n_frames=8
):

    cap = cv2.VideoCapture(
        str(video_path)
    )

    if not cap.isOpened():

        return []

    frame_count = int(
        cap.get(
            cv2.CAP_PROP_FRAME_COUNT
        )
    )

    target_indices = (
        uniform_frame_indices(
            frame_count,
            n_frames
        )
    )

    target_set = set(
        target_indices
    )

    frames = []

    current_index = 0

    while True:

        ok, frame = cap.read()

        if not ok:
            break

        if (
            current_index
            in
            target_set
        ):

            frames.append(
                (
                    current_index,
                    frame
                )
            )

            if (
                len(frames)
                ==
                len(target_indices)
            ):

                break

        current_index += 1

    cap.release()

    return frames


# =============================================================================
# 13. TEMPORAL STABILITY
# =============================================================================

def compute_temporal_stability(
    frame_boxes,
    width,
    height
):

    if not frame_boxes:

        return 0.0

    valid_fraction = (
        sum(
            b is not None
            for b in frame_boxes
        )
        /
        len(frame_boxes)
    )

    distances = []

    for i in range(
        1,
        len(frame_boxes)
    ):

        d = center_distance(
            frame_boxes[i - 1],
            frame_boxes[i],
            width,
            height
        )

        if not np.isnan(d):

            distances.append(
                d
            )

    if not distances:

        return float(
            valid_fraction
        )

    mean_distance = float(
        np.mean(
            distances
        )
    )

    motion_stability = math.exp(
        -3.0
        *
        mean_distance
    )

    stability = (
        0.7
        *
        motion_stability
        +
        0.3
        *
        valid_fraction
    )

    return float(
        np.clip(
            stability,
            0.0,
            1.0
        )
    )


# =============================================================================
# 14. BENCHMARK INITIALIZATION
# =============================================================================

all_frame_results = []

video_results = []

detector_errors = []

print(
    "\n" + "=" * 80
)

print(
    "BEGIN FINAL DETECTOR BENCHMARK"
)

print(
    "=" * 80
)

print(
    f"\nVideos: {len(selection_manifest)}"
)

print(
    f"Frames/video: {FRAMES_PER_VIDEO}"
)

print(
    "\nIMPORTANT: MediaPipe Landmarker is recreated "
    "for every video."
)


# =============================================================================
# 15. PROCESS VIDEOS
# =============================================================================

for video_number, row in enumerate(
    selection_manifest.itertuples(
        index=False
    ),
    start=1
):

    uid = str(
        row.uid
    )

    video_path = Path(
        str(
            row.video_path
        )
    )

    frames = read_uniform_frames(
        video_path,
        FRAMES_PER_VIDEO
    )

    if not frames:

        detector_errors.append({

            "uid":
                uid,

            "detector":
                "VIDEO_DECODER",

            "frame_index":
                -1,

            "error_type":
                "decode_failure",

            "error_message":
                "No frames decoded."
        })

        print(
            f"[{video_number:03d}/"
            f"{len(selection_manifest)}] "
            f"DECODE FAILURE: {uid}"
        )

        continue


    height, width = (
        frames[0][1].shape[:2]
    )


    # =========================================================================
    # CRITICAL FIX:
    # Create a fresh MediaPipe detector for THIS VIDEO ONLY.
    # =========================================================================

    try:

        mediapipe_landmarker = (
            create_mediapipe_landmarker()
        )

    except Exception as e:

        detector_errors.append({

            "uid":
                uid,

            "detector":
                "MediaPipe_Hands",

            "frame_index":
                -1,

            "error_type":
                type(e).__name__,

            "error_message":
                str(e)
        })

        mediapipe_landmarker = None


    # =========================================================================
    # FRAME PROCESSING
    # =========================================================================

    for detector_name in [
        "MediaPipe_Hands",
        "OpenCV_Skin_Contour"
    ]:

        frame_boxes = []

        detection_count = 0

        one_hand_count = 0

        two_hand_count = 0

        invalid_count = 0

        error_count = 0

        confidences = []

        roi_areas = []

        latencies = []


        for frame_position, (
            frame_index,
            frame
        ) in enumerate(frames):

            # -----------------------------------------------------------------
            # Timestamp is local to THIS video.
            #
            # Since the MediaPipe landmarker itself is recreated per video,
            # timestamps can safely start at zero.
            # -----------------------------------------------------------------

            timestamp_ms = (
                frame_position
                *
                100
            )

            start_time = (
                time.perf_counter()
            )

            exception_message = None


            # -----------------------------------------------------------------
            # RUN DETECTOR
            # -----------------------------------------------------------------

            try:

                if (
                    detector_name
                    ==
                    "MediaPipe_Hands"
                ):

                    if (
                        mediapipe_landmarker
                        is None
                    ):

                        raise RuntimeError(
                            "MediaPipe landmarker "
                            "was not initialized "
                            "for this video."
                        )

                    result = (
                        detect_mediapipe(
                            mediapipe_landmarker,
                            frame,
                            timestamp_ms
                        )
                    )

                else:

                    result = (
                        detect_opencv_skin(
                            frame
                        )
                    )

            except Exception as e:

                error_count += 1

                exception_message = (
                    str(e)
                )

                detector_errors.append({

                    "uid":
                        uid,

                    "detector":
                        detector_name,

                    "frame_index":
                        int(frame_index),

                    "error_type":
                        type(e).__name__,

                    "error_message":
                        str(e)
                })

                result = {

                    "boxes": [],

                    "confidence": 0.0,

                    "num_hands": 0,

                    "roi_box": None
                }


            elapsed_ms = (
                time.perf_counter()
                -
                start_time
            ) * 1000.0

            latencies.append(
                elapsed_ms
            )


            # -----------------------------------------------------------------
            # EXTRACT RESULT
            # -----------------------------------------------------------------

            roi_box = result[
                "roi_box"
            ]

            num_hands = result[
                "num_hands"
            ]

            confidence = result[
                "confidence"
            ]

            area_fraction = (
                box_area_fraction(
                    roi_box,
                    width,
                    height
                )
            )


            # -----------------------------------------------------------------
            # ROI VALIDITY
            # -----------------------------------------------------------------

            valid_roi = (

                roi_box is not None

                and

                area_fraction
                >=
                MIN_ROI_AREA_FRACTION

                and

                area_fraction
                <=
                MAX_ROI_AREA_FRACTION
            )


            if valid_roi:

                detection_count += 1

                if num_hands == 1:

                    one_hand_count += 1

                elif num_hands >= 2:

                    two_hand_count += 1

                confidences.append(
                    confidence
                )

                roi_areas.append(
                    area_fraction
                )

            else:

                invalid_count += 1

                roi_box = None

                area_fraction = 0.0


            frame_boxes.append(
                roi_box
            )


            all_frame_results.append({

                "uid":
                    uid,

                "gloss":
                    str(row.gloss),

                "detector":
                    detector_name,

                "frame_index":
                    int(frame_index),

                "width":
                    int(width),

                "height":
                    int(height),

                "detected":
                    int(valid_roi),

                "num_hands":
                    int(num_hands),

                "confidence":
                    float(confidence),

                "roi_area_fraction":
                    float(area_fraction),

                "latency_ms":
                    float(elapsed_ms),

                "error":
                    exception_message
            })


        # =====================================================================
        # VIDEO-LEVEL METRICS
        # =====================================================================

        n_frames_actual = len(
            frames
        )

        valid_rate = (
            detection_count
            /
            max(
                n_frames_actual,
                1
            )
        )

        one_hand_rate = (
            one_hand_count
            /
            max(
                n_frames_actual,
                1
            )
        )

        two_hand_rate = (
            two_hand_count
            /
            max(
                n_frames_actual,
                1
            )
        )

        invalid_rate = (
            invalid_count
            /
            max(
                n_frames_actual,
                1
            )
        )

        error_rate = (
            error_count
            /
            max(
                n_frames_actual,
                1
            )
        )

        avg_confidence = (

            float(
                np.mean(
                    confidences
                )
            )

            if confidences

            else 0.0
        )

        mean_roi_area = (

            float(
                np.mean(
                    roi_areas
                )
            )

            if roi_areas

            else 0.0
        )

        temporal_stability = (
            compute_temporal_stability(
                frame_boxes,
                width,
                height
            )
        )

        mean_latency = (

            float(
                np.mean(
                    latencies
                )
            )

            if latencies

            else np.nan
        )


        video_results.append({

            "uid":
                uid,

            "gloss":
                str(row.gloss),

            "detector":
                detector_name,

            "frames_evaluated":
                n_frames_actual,

            "valid_detection_rate":
                valid_rate,

            "one_hand_rate":
                one_hand_rate,

            "two_hand_rate":
                two_hand_rate,

            "invalid_rate":
                invalid_rate,

            "error_rate":
                error_rate,

            "average_confidence":
                avg_confidence,

            "mean_roi_area_fraction":
                mean_roi_area,

            "temporal_stability":
                temporal_stability,

            "mean_latency_ms":
                mean_latency
        })


    # =========================================================================
    # CLOSE MEDIA PIPE INSTANCE FOR THIS VIDEO
    # =========================================================================

    if (
        mediapipe_landmarker
        is not None
    ):

        try:

            mediapipe_landmarker.close()

        except Exception:

            pass


    # =========================================================================
    # PROGRESS
    # =========================================================================

    if (

        video_number == 1

        or

        video_number % 10 == 0

        or

        video_number
        ==
        len(selection_manifest)

    ):

        print(
            f"[{video_number:03d}/"
            f"{len(selection_manifest)}] "
            f"{uid}"
        )


# =============================================================================
# 16. SAVE ERROR LOG
# =============================================================================

error_df = pd.DataFrame(
    detector_errors
)

error_df.to_csv(
    ERROR_LOG_CSV,
    index=False
)

print(
    "\nDetector error log saved:"
)

print(
    ERROR_LOG_CSV
)

print(
    "Total recorded errors:",
    len(error_df)
)


# =============================================================================
# 17. SAVE FRAME-LEVEL RESULTS
# =============================================================================

frame_results_df = pd.DataFrame(
    all_frame_results
)

frame_results_df.to_csv(
    FRAME_RESULTS_CSV,
    index=False
)

print(
    "\nFrame-level results saved:"
)

print(
    FRAME_RESULTS_CSV
)


# =============================================================================
# 18. AGGREGATE DETECTOR RESULTS
# =============================================================================

video_results_df = pd.DataFrame(
    video_results
)

if video_results_df.empty:

    raise RuntimeError(
        "No detector results were generated."
    )


comparison = (

    video_results_df

    .groupby(
        "detector"
    )

    .agg(

        videos_evaluated=(
            "uid",
            "nunique"
        ),

        valid_detection_rate=(
            "valid_detection_rate",
            "mean"
        ),

        one_hand_rate=(
            "one_hand_rate",
            "mean"
        ),

        two_hand_rate=(
            "two_hand_rate",
            "mean"
        ),

        invalid_rate=(
            "invalid_rate",
            "mean"
        ),

        error_rate=(
            "error_rate",
            "mean"
        ),

        average_confidence=(
            "average_confidence",
            "mean"
        ),

        mean_roi_area_fraction=(
            "mean_roi_area_fraction",
            "mean"
        ),

        temporal_stability=(
            "temporal_stability",
            "mean"
        ),

        mean_latency_ms=(
            "mean_latency_ms",
            "mean"
        )
    )

    .reset_index()
)


# =============================================================================
# 19. NORMALIZATION HELPERS
# =============================================================================

def minmax_positive(series):

    series = pd.Series(
        series,
        dtype=float
    )

    if (
        series.max()
        ==
        series.min()
    ):

        return pd.Series(
            np.ones(
                len(series)
            ),
            index=series.index
        )

    return (
        (
            series
            -
            series.min()
        )
        /
        (
            series.max()
            -
            series.min()
        )
    )


def minmax_negative(series):

    series = pd.Series(
        series,
        dtype=float
    )

    if (
        series.max()
        ==
        series.min()
    ):

        return pd.Series(
            np.ones(
                len(series)
            ),
            index=series.index
        )

    return (
        (
            series.max()
            -
            series
        )
        /
        (
            series.max()
            -
            series.min()
        )
    )


# =============================================================================
# 20. SELECTION SCORES
# =============================================================================

comparison[
    "score_detection"
] = minmax_positive(
    comparison[
        "valid_detection_rate"
    ]
)

comparison[
    "score_stability"
] = minmax_positive(
    comparison[
        "temporal_stability"
    ]
)

comparison[
    "score_confidence"
] = minmax_positive(
    comparison[
        "average_confidence"
    ]
)

comparison[
    "score_two_hand"
] = minmax_positive(
    comparison[
        "two_hand_rate"
    ]
)

comparison[
    "score_speed"
] = minmax_negative(
    comparison[
        "mean_latency_ms"
    ]
)


# =============================================================================
# ROI AREA SCORE
# =============================================================================

TARGET_ROI_AREA = 0.20

comparison[
    "roi_area_score"
] = (

    1.0

    -

    (
        abs(
            comparison[
                "mean_roi_area_fraction"
            ]
            -
            TARGET_ROI_AREA
        )
        /
        TARGET_ROI_AREA
    )
)

comparison[
    "roi_area_score"
] = (
    comparison[
        "roi_area_score"
    ]
    .clip(
        0.0,
        1.0
    )
)


# =============================================================================
# ERROR-FREE SCORE
# =============================================================================

comparison[
    "score_error_free"
] = (
    1.0
    -
    comparison[
        "error_rate"
    ]
).clip(
    0.0,
    1.0
)


# =============================================================================
# 21. FINAL WEIGHTED SCORE
# =============================================================================
#
# Detection reliability       30%
# Temporal stability          20%
# Confidence                  10%
# Two-hand handling           10%
# ROI area                    10%
# Speed                       10%
# Error-free operation        10%
# =============================================================================

comparison[
    "overall_selection_score"
] = (

    0.30
    *
    comparison[
        "score_detection"
    ]

    +

    0.20
    *
    comparison[
        "score_stability"
    ]

    +

    0.10
    *
    comparison[
        "score_confidence"
    ]

    +

    0.10
    *
    comparison[
        "score_two_hand"
    ]

    +

    0.10
    *
    comparison[
        "roi_area_score"
    ]

    +

    0.10
    *
    comparison[
        "score_speed"
    ]

    +

    0.10
    *
    comparison[
        "score_error_free"
    ]
)


comparison = (
    comparison
    .sort_values(
        "overall_selection_score",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)

comparison["rank"] = (
    np.arange(
        1,
        len(comparison) + 1
    )
)


# =============================================================================
# 22. SAVE COMPARISON
# =============================================================================

comparison.to_csv(
    RESULTS_CSV,
    index=False
)

print(
    "\n" + "=" * 80
)

print(
    "FINAL DETECTOR COMPARISON"
)

print(
    "=" * 80
)

display_columns = [

    "rank",

    "detector",

    "videos_evaluated",

    "valid_detection_rate",

    "one_hand_rate",

    "two_hand_rate",

    "invalid_rate",

    "error_rate",

    "average_confidence",

    "mean_roi_area_fraction",

    "temporal_stability",

    "mean_latency_ms",

    "overall_selection_score"
]

print(
    comparison[
        display_columns
    ].to_string(
        index=False
    )
)


# =============================================================================
# 23. CHECK MEDIAPIPE STATUS
# =============================================================================

mp_row = comparison[
    comparison[
        "detector"
    ]
    ==
    "MediaPipe_Hands"
]

if not mp_row.empty:

    mp_valid = float(
        mp_row.iloc[0][
            "valid_detection_rate"
        ]
    )

    mp_errors = float(
        mp_row.iloc[0][
            "error_rate"
        ]
    )

    print(
        "\n" + "=" * 80
    )

    print(
        "MEDIAPIPE STATUS"
    )

    print(
        "=" * 80
    )

    print(
        f"Valid ROI rate: "
        f"{mp_valid:.4f}"
    )

    print(
        f"Error rate: "
        f"{mp_errors:.4f}"
    )

    if mp_errors == 0:

        print(
            "\nMediaPipe execution errors: NONE"
        )

        print(
            "The MediaPipe result is now "
            "scientifically interpretable."
        )

    else:

        print(
            "\nWARNING:"
        )

        print(
            "MediaPipe still has execution errors."
        )

        print(
            "Inspect the error log before "
            "interpreting detector performance."
        )


# =============================================================================
# 24. PROVISIONAL WINNER
# =============================================================================

winner = comparison.iloc[0]

selected_detector = str(
    winner[
        "detector"
    ]
)

selected_score = float(
    winner[
        "overall_selection_score"
    ]
)

print(
    "\n" + "=" * 80
)

print(
    "PROVISIONAL DETECTOR SELECTION"
)

print(
    "=" * 80
)

print(
    "\nSelected method:",
    selected_detector
)

print(
    "Selection score:",
    f"{selected_score:.4f}"
)

print(
    "Valid detection rate:",
    f"{winner['valid_detection_rate']:.4f}"
)

print(
    "Temporal stability:",
    f"{winner['temporal_stability']:.4f}"
)

print(
    "Mean ROI area:",
    f"{winner['mean_roi_area_fraction']:.4f}"
)

print(
    "Mean latency:",
    f"{winner['mean_latency_ms']:.3f} ms/frame"
)


# =============================================================================
# 25. VISUAL SANITY CHECK
# =============================================================================

print(
    "\n" + "=" * 80
)

print(
    "GENERATING VISUAL SANITY CHECK"
)

print(
    "=" * 80
)

visual_subset = (
    selection_manifest
    .head(
        N_VISUAL_VIDEOS
    )
)

visual_records = []


for video_number, row in enumerate(
    visual_subset.itertuples(
        index=False
    ),
    start=1
):

    uid = str(
        row.uid
    )

    video_path = Path(
        str(
            row.video_path
        )
    )

    frames = read_uniform_frames(
        video_path,
        n_frames=4
    )

    if not frames:
        continue


    selected_frame_index, frame = (
        frames[
            len(frames) // 2
        ]
    )


    # -------------------------------------------------------------------------
    # Fresh MediaPipe instance for visual video too.
    # -------------------------------------------------------------------------

    try:

        visual_mp = (
            create_mediapipe_landmarker()
        )

    except Exception as e:

        visual_mp = None

        detector_errors.append({

            "uid":
                uid,

            "detector":
                "MediaPipe_Hands_VISUAL",

            "frame_index":
                int(selected_frame_index),

            "error_type":
                type(e).__name__,

            "error_message":
                str(e)
        })


    for detector_name in [
        "MediaPipe_Hands",
        "OpenCV_Skin_Contour"
    ]:

        try:

            if detector_name == "MediaPipe_Hands":

                if visual_mp is None:

                    raise RuntimeError(
                        "MediaPipe visual "
                        "landmarker unavailable."
                    )

                result = (
                    detect_mediapipe(
                        visual_mp,
                        frame,
                        100
                    )
                )

            else:

                result = (
                    detect_opencv_skin(
                        frame
                    )
                )

        except Exception as e:

            visual_records.append({

                "uid":
                    uid,

                "detector":
                    detector_name,

                "frame_index":
                    selected_frame_index,

                "status":
                    "ERROR",

                "error":
                    str(e)
            })

            continue


        display_frame = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )

        fig, ax = plt.subplots(
            figsize=(8, 6)
        )

        ax.imshow(
            display_frame
        )

        ax.set_title(
            f"{detector_name}\nUID: {uid}"
        )

        roi_box = result[
            "roi_box"
        ]

        if roi_box is not None:

            x1, y1, x2, y2 = (
                roi_box
            )

            rect = plt.Rectangle(
                (
                    x1,
                    y1
                ),
                x2 - x1,
                y2 - y1,
                fill=False,
                linewidth=2
            )

            ax.add_patch(
                rect
            )

            area = (
                box_area_fraction(
                    roi_box,
                    frame.shape[1],
                    frame.shape[0]
                )
            )

            ax.text(
                x1,
                max(
                    y1 - 5,
                    5
                ),
                (
                    f"hands={result['num_hands']} | "
                    f"area={area:.2f}"
                ),
                fontsize=9
            )

            status = "VALID"

        else:

            status = "NO_ROI"

        ax.axis(
            "off"
        )

        safe_uid = (
            uid
            .replace(
                "/",
                "_"
            )
            .replace(
                "\\",
                "_"
            )
        )

        filename = (
            VISUAL_DIR
            /
            f"{video_number:02d}_"
            f"{safe_uid}_"
            f"{detector_name}.png"
        )

        plt.tight_layout()

        plt.savefig(
            filename,
            dpi=120,
            bbox_inches="tight"
        )

        plt.close(
            fig
        )

        visual_records.append({

            "uid":
                uid,

            "detector":
                detector_name,

            "frame_index":
                selected_frame_index,

            "status":
                status,

            "visual_file":
                str(filename)
        })


    if visual_mp is not None:

        try:

            visual_mp.close()

        except Exception:

            pass


visual_results_df = pd.DataFrame(
    visual_records
)

visual_results_csv = (
    OUTPUT_ROOT
    /
    "stage5_2_visual_sanity_manifest.csv"
)

visual_results_df.to_csv(
    visual_results_csv,
    index=False
)

print(
    "\nVisual sanity-check images saved:"
)

print(
    VISUAL_DIR
)

print(
    "Visual manifest:"
)

print(
    visual_results_csv
)


# =============================================================================
# 26. SAVE FINAL PROTOCOL
# =============================================================================

protocol = {

    "stage":
        "5.2",

    "title":
        "Hand Detector / ROI Generator Selection",

    "version":
        "final_corrected_per_video_mediapipe",

    "project_root":
        str(PROJECT_ROOT),

    "input_manifest":
        str(MANIFEST_PATH),

    "selection_manifest":
        str(SELECTION_MANIFEST_PATH),

    "selection_videos":
        int(N_SELECTION_VIDEOS),

    "frames_per_video":
        int(FRAMES_PER_VIDEO),

    "candidate_methods": [

        "MediaPipe_Hands",

        "OpenCV_Skin_Contour"
    ],

    "mediapipe_timestamp_strategy":
        (
            "A fresh MediaPipe HandLandmarker instance "
            "is created for each independent video. "
            "Timestamps restart from zero for each video "
            "and increase monotonically within that video."
        ),

    "evaluation_metrics": [

        "valid_detection_rate",

        "one_hand_rate",

        "two_hand_rate",

        "invalid_rate",

        "error_rate",

        "average_confidence",

        "mean_roi_area_fraction",

        "temporal_stability",

        "mean_latency_ms"
    ],

    "selection_weights": {

        "detection_reliability":
            0.30,

        "temporal_stability":
            0.20,

        "confidence":
            0.10,

        "two_hand_handling":
            0.10,

        "roi_area":
            0.10,

        "speed":
            0.10,

        "error_free_operation":
            0.10
    },

    "roi_validity": {

        "minimum_area_fraction":
            MIN_ROI_AREA_FRACTION,

        "maximum_area_fraction":
            MAX_ROI_AREA_FRACTION,

        "target_area_fraction":
            TARGET_ROI_AREA
    },

    "ground_truth_limitation":
        (
            "The current CISLR protocol does not contain "
            "hand bounding-box ground truth. Therefore "
            "detector localization IoU is not claimed."
        ),

    "visual_sanity_check": {

        "videos":
            int(N_VISUAL_VIDEOS),

        "purpose":
            (
                "Qualitative verification that generated "
                "ROIs correspond to signer hand regions."
            )
    },

    "raw_videos_modified":
        False,

    "stage4_artifacts_modified":
        False,

    "stage5_1_artifacts_modified":
        False,

    "selected_detector":
        selected_detector,

    "selected_score":
        selected_score
}

with open(
    PROTOCOL_JSON,
    "w"
) as f:

    json.dump(
        protocol,
        f,
        indent=2
    )


# =============================================================================
# 27. SAVE SUMMARY
# =============================================================================

summary = {

    "stage":
        "5.2",

    "status":
        "COMPLETE_PENDING_VISUAL_REVIEW",

    "selection_videos":
        int(
            len(selection_manifest)
        ),

    "selected_detector":
        selected_detector,

    "selection_score":
        selected_score,

    "selected_metrics": {

        "valid_detection_rate":
            float(
                winner[
                    "valid_detection_rate"
                ]
            ),

        "one_hand_rate":
            float(
                winner[
                    "one_hand_rate"
                ]
            ),

        "two_hand_rate":
            float(
                winner[
                    "two_hand_rate"
                ]
            ),

        "invalid_rate":
            float(
                winner[
                    "invalid_rate"
                ]
            ),

        "error_rate":
            float(
                winner[
                    "error_rate"
                ]
            ),

        "average_confidence":
            float(
                winner[
                    "average_confidence"
                ]
            ),

        "mean_roi_area_fraction":
            float(
                winner[
                    "mean_roi_area_fraction"
                ]
            ),

        "temporal_stability":
            float(
                winner[
                    "temporal_stability"
                ]
            ),

        "mean_latency_ms":
            float(
                winner[
                    "mean_latency_ms"
                ]
            )
    },

    "visual_review_required":
        True,

    "next_stage":
        (
            "Stage 5.3 — Hand ROI Extraction "
            "after visual sanity-check approval."
        )
}

with open(
    SUMMARY_JSON,
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )


# =============================================================================
# 28. FINAL REPORT
# =============================================================================

print(
    "\n" + "=" * 80
)

print(
    "STAGE 5.2 FINAL CORRECTED RUN COMPLETE"
)

print(
    "=" * 80
)

print(
    "\nArtifacts:"
)

print(
    "1.",
    SELECTION_MANIFEST_PATH
)

print(
    "2.",
    FRAME_RESULTS_CSV
)

print(
    "3.",
    RESULTS_CSV
)

print(
    "4.",
    ERROR_LOG_CSV
)

print(
    "5.",
    PROTOCOL_JSON
)

print(
    "6.",
    SUMMARY_JSON
)

print(
    "7.",
    visual_results_csv
)

print(
    "8.",
    VISUAL_DIR
)

print(
    "\nSelected detector:",
    selected_detector
)

print(
    "\nIMPORTANT:"
)

print(
    "Stage 5.2 is now numerically complete."
)

print(
    "However, review the visual_sanity_check/"
    " images before freezing the detector."
)

print(
    "\nDO NOT start Stage 5.3 until the ROIs "
    "have been visually confirmed."
)

print(
    "\nDO NOT rerun:"
)

print(
    "  Stage 4.1 → Stage 4.9"
)

print(
    "  Stage 5.1"
)

print(
    "  MobileNetV3 training"
)

print(
    "  Official retrieval evaluation"
)

print(
    "\nRaw CISLR videos remain untouched."
)

print(
    "=" * 80
)

CISLR — STAGE 5.2
FINAL CORRECTED HAND DETECTOR / ROI GENERATOR SELECTION

Project root: /home/dipshikha/Music/cao/CISLR_RESEARCH

Manifest shape: (7049, 14)
Unique videos: 7049
Videos found: 7049
Videos missing: 0

Fixed detector-selection subset: 200 videos
Saved: /home/dipshikha/Music/cao/CISLR_RESEARCH/audit/stage5_hand_roi/detector_selection/stage5_2_selection_video_manifest.csv
Subset verification: IDENTICAL

Checking MediaPipe...
MediaPipe version: 1.0.1

Hand Landmarker model already exists.
Model: /home/dipshikha/Music/cao/CISLR_RESEARCH/audit/stage5_hand_roi/detector_selection/models/hand_landmarker.task


W0000 00:00:1789500964.265762   49615 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1789500964.280432   49615 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


MediaPipe Hand Landmarker initialization: PASS

BEGIN FINAL DETECTOR BENCHMARK

Videos: 200
Frames/video: 8

IMPORTANT: MediaPipe Landmarker is recreated for every video.


W0000 00:00:1789500964.479217   49652 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1789500964.492322   49652 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


[001/200] T4VLu2P-X9s_1


...

[170/200] S1eFafNFNOE


W0000 00:00:1789501036.710883   56032 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1789501036.717855   56033 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1789501037.033498   56068 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1789501037.042219   56070 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1789501039.209841   56118 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1789501039.219899   56118 inference_feedback_manager.cc:121] Feedback manager 

[180/200] CXOPG4pJxDI_1


W0000 00:00:1789501044.235500   56416 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1789501044.241859   56417 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1789501044.679218   56452 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1789501044.686044   56453 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1789501044.980985   56489 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1789501044.987465   56491 inference_feedback_manager.cc:121] Feedback manager 

KeyboardInterrupt: 